### Ejercicio 1 – Precio promedio por barrio

Enunciado:

Agrupa por barrio y calcula el precio medio de las viviendas. ¿Qué barrio es el más caro en promedio?

In [2]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv('Dataset_vivienda_en_Madrid.csv')

In [4]:
df.head(10)

,barrio,metros_cuadrados,habitaciones,baños,precio,latitud,longitud
0,Salamanca,138,2,3,580263,40.431979,-3.680183
1,Arganzuela,113,1,3,461554,40.400009,-3.695814
2,Chamartín,118,5,1,573500,40.463564,-3.677655
3,Villa de Vallecas,106,5,1,514179,40.366842,-3.603498
4,Tetuán,139,5,1,571224,40.457287,-3.704847
5,Arganzuela,116,1,3,509474,40.398071,-3.697330
6,Chamartín,98,3,3,384454,40.464462,-3.679432
7,San Blas,110,4,3,476003,40.431858,-3.620340
8,Moratalaz,143,4,3,615712,40.405094,-3.655796
9,San Blas,96,2,1,345379,40.435213,-3.622583


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   barrio            250 non-null    object 
 1   metros_cuadrados  250 non-null    int64  
 2   habitaciones      250 non-null    int64  
 3   baños             250 non-null    int64  
 4   precio            250 non-null    int64  
 5   latitud           250 non-null    float64
 6   longitud          250 non-null    float64
dtypes: float64(2), int64(4), object(1)
memory usage: 13.8+ KB


In [7]:
df_agrupado=df.groupby('barrio')['precio'].mean().sort_values(ascending=False).round()
df_agrupado

barrio
Arganzuela            532983.0
Carabanchel           525438.0
Salamanca             510747.0
Latina                508567.0
Puente de Vallecas    507814.0
Moratalaz             498366.0
Usera                 496346.0
Retiro                485115.0
Villa de Vallecas     483567.0
Tetuán                478553.0
Chamartín             465585.0
Centro                464953.0
Hortaleza             449413.0
Moncloa               408381.0
San Blas              394233.0
Name: precio, dtype: float64

### Ejercicio 2 – Precio medio por metro cuadrado

Enunciado:

Agrega una nueva columna precio_m2 y calcula el promedio de este valor por barrio. Usa groupby + apply.

In [9]:
df['precio_m2']=(df['precio']/df['metros_cuadrados']).round()
df.head(3)


,barrio,metros_cuadrados,habitaciones,baños,precio,latitud,longitud,precio_m2
0,Salamanca,138,2,3,580263,40.431979,-3.680183,4205.0
1,Arganzuela,113,1,3,461554,40.400009,-3.695814,4085.0
2,Chamartín,118,5,1,573500,40.463564,-3.677655,4860.0


In [13]:
df.groupby('barrio')['precio_m2'].mean().round().sort_values(ascending=False)

barrio
San Blas              4824.0
Moncloa               4474.0
Retiro                4447.0
Tetuán                4438.0
Moratalaz             4373.0
Centro                4366.0
Usera                 4343.0
Latina                4333.0
Salamanca             4277.0
Villa de Vallecas     4260.0
Carabanchel           4255.0
Hortaleza             4254.0
Puente de Vallecas    4222.0
Arganzuela            4209.0
Chamartín             4141.0
Name: precio_m2, dtype: float64

### Ejercicio 3 – Clasificación por cuartiles de precio

Enunciado:

Agrupa por barrio, calcula el precio promedio y asigna a cada barrio un cuartil (Q1–Q4) según el valor medio.

In [15]:
precio_prom= df.groupby('barrio')['precio'].mean().round()
precio_prom

barrio
Arganzuela            532983.0
Carabanchel           525438.0
Centro                464953.0
Chamartín             465585.0
Hortaleza             449413.0
Latina                508567.0
Moncloa               408381.0
Moratalaz             498366.0
Puente de Vallecas    507814.0
Retiro                485115.0
Salamanca             510747.0
San Blas              394233.0
Tetuán                478553.0
Usera                 496346.0
Villa de Vallecas     483567.0
Name: precio, dtype: float64

In [16]:
cuartil= pd.qcut(precio_prom , q=4, labels=['Q1','Q2','Q3','Q4'])
cuartil



barrio
Arganzuela            Q4
Carabanchel           Q4
Centro                Q1
Chamartín             Q2
Hortaleza             Q1
Latina                Q4
Moncloa               Q1
Moratalaz             Q3
Puente de Vallecas    Q3
Retiro                Q2
Salamanca             Q4
San Blas              Q1
Tetuán                Q2
Usera                 Q3
Villa de Vallecas     Q2
Name: precio, dtype: category
Categories (4, object): ['Q1' < 'Q2' < 'Q3' < 'Q4']

In [17]:
df['Cuartil_barrio']=df['barrio'].map(cuartil)
df.head()

,barrio,metros_cuadrados,habitaciones,baños,precio,latitud,longitud,precio_m2,Cuartil_barrio
0,Salamanca,138,2,3,580263,40.431979,-3.680183,4205.0,Q4
1,Arganzuela,113,1,3,461554,40.400009,-3.695814,4085.0,Q4
2,Chamartín,118,5,1,573500,40.463564,-3.677655,4860.0,Q2
3,Villa de Vallecas,106,5,1,514179,40.366842,-3.603498,4851.0,Q2
4,Tetuán,139,5,1,571224,40.457287,-3.704847,4110.0,Q2


### Ejercicio 4 – Detectar viviendas “anómalas” en su barrio

Enunciado:

Una vivienda es “anómala” si su precio se aleja más de 2 desviaciones estándar del promedio de su barrio. Crea una nueva columna que indique esto.

In [23]:
def detectar_anomalias(anomalia):
    media= anomalia['precio'].mean()
    std= anomalia['precio'].std()
    return anomalia['precio'].apply(lambda x: abs(x-media)>2*std)



In [24]:
df['anomalia'] = df.groupby('barrio').apply(detectar_anomalias).reset_index(drop=True)
df_anomalos=df[df['anomalia']== True]
df_anomalos

C:\Users\josemanuel.trivino\AppData\Local\Temp\ipykernel_19156\3570934875.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['anomalia'] = df.groupby('barrio').apply(detectar_anomalias).reset_index(drop=True)


,barrio,metros_cuadrados,habitaciones,baños,precio,latitud,longitud,precio_m2,Cuartil_barrio,anomalia
69,Hortaleza,148,2,1,567510,40.477466,-3.641553,3835.0,Q1,True
81,Usera,119,4,3,554275,40.381551,-3.707143,4658.0,Q3,True
110,Moratalaz,67,2,2,310383,40.404577,-3.653694,4633.0,Q3,True


### Ejercicio 5 – Mapa con folium: precios por color

Enunciado:

Crea un mapa con folium donde cada vivienda tenga un color según su precio:

Verde: <300.000 €

Naranja: 300.000–600.000 €

Rojo: >600.000 €

In [31]:
import folium

def color_precio(precio):
    if precio < 300000 :
        return 'green'
    elif precio <600000:
        return'orange'
    else:
        return 'Red'


In [34]:
mapa = folium.Map(location=[40.4168 , -3.7038], zoom_start=12)

In [37]:
for _, fila in df.iterrows():
    folium.CircleMarker(
        location=[fila['latitud'], fila['longitud']],
        radius= 5,
        color=color_precio(fila['precio']),
        fill=True,
        fill_opacity=0.6,
        popup=f"Barrio:{fila['barrio']}-Precio:{fila['precio']}€-Hab:{fila['habitaciones']}-Baños:{fila['baños']}-Mts2:{fila['metros_cuadrados']}"
    ).add_to(mapa)
mapa.save('Mapa_colores_precio.html')    


### Ejercicio 6 – Mapa de calor de viviendas caras

Enunciado:

Crea un mapa de calor con folium.plugins.HeatMap que indique la concentración de viviendas con precio superior a 600.000 €.

In [38]:
from folium.plugins import HeatMap

mapa_calor = folium.Map(location=[40.4168, -3.7038], zoom_start=12)#localiacion madrid
data_heat = df[df["precio"] > 600000][["latitud", "longitud"]].values.tolist()
data_heat



[[40.40509371037048, -3.6557962871090783],
 [40.39346533776347, -3.745065258255051],
 [40.43061025608132, -3.62352137559689],
 [40.4168787689252, -3.702800252699016],
 [40.41550571206243, -3.6844877490284578],
 [40.42958116465288, -3.6763344945590872],
 [40.46319385200736, -3.679090575020849],
 [40.45772433248764, -3.702267228033784],
 [40.46329305275488, -3.678363425635328],
 [40.40991038480684, -3.657152221946781],
 [40.4166760050241, -3.704918745052863],
 [40.40568407148945, -3.657067324803368],
 [40.40830985536648, -3.651976618400501],
 [40.40136833397701, -3.6962975273361134],
 [40.43278427928036, -3.624303881726642],
 [40.43033998521466, -3.678412858985231],
 [40.40995252650508, -3.655715250092893],
 [40.385230594761374, -3.7049831317825257],
 [40.39593584598645, -3.7484571924696057],
 [40.381397111164354, -3.708735456239956],
 [40.38200639394892, -3.7471069152421],
 [40.393408718695895, -3.747863788477262],
 [40.41501304631147, -3.7012238250536895],
 [40.39981594489424, -3.69443

In [39]:
HeatMap(data_heat, radius=15, blur=10).add_to(mapa_calor)
mapa_calor.save("mapa_calor_viviendas_caras.html")

### Ejercicio 7 – Vivienda más barata por barrio

Enunciado:

Encuentra la vivienda más barata de cada barrio (precio mínimo). Muestra su ubicación en un mapa.

In [40]:

viviendas_baratas = df.loc[df.groupby("barrio")["precio"].idxmin()]
viviendas_baratas



,barrio,metros_cuadrados,habitaciones,baños,precio,latitud,longitud,precio_m2,Cuartil_barrio,anomalia
49,Arganzuela,68,1,1,280321,40.399485,-3.696941,4122.0,Q4,False
113,Carabanchel,65,1,1,257049,40.385351,-3.746175,3955.0,Q4,False
221,Centro,50,1,3,249504,40.415773,-3.703681,4990.0,Q1,False
42,Chamartín,43,1,1,193223,40.464675,-3.676431,4494.0,Q2,False
84,Hortaleza,54,1,2,191782,40.472032,-3.639939,3552.0,Q1,False
143,Latina,42,1,2,183732,40.393478,-3.743338,4375.0,Q4,False
60,Moncloa,46,3,1,170900,40.431853,-3.717174,3715.0,Q1,False
27,Moratalaz,50,5,1,247324,40.407457,-3.655199,4946.0,Q3,False
98,Puente de Vallecas,71,2,2,323315,40.392462,-3.659745,4554.0,Q3,False
209,Retiro,54,2,1,200475,40.415968,-3.685210,3712.0,Q2,False


In [41]:
mapa_min = folium.Map(location=[40.4168, -3.7038], zoom_start=12)
for _, row in viviendas_baratas.iterrows():
    folium.Marker(
        location=[row["latitud"], row["longitud"]],
        popup=f"{row['barrio']} - {row['precio']} €",
        icon=folium.Icon(color="blue", icon="home")
    ).add_to(mapa_min)

mapa_min.save("mapa_viviendas_baratas.html")